# Example 11.3.1.2 / 1: Use a Special Function in an Objective

Companion notebook for **11.3.1.2 GSL Special Functions** in the AMPL lab Documentation, section "Use a Special Function in an Objective". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Use a Special Function in an Objective

A special function can appear in a parameter, a constraint or an objective. Inside an objective, a
solver that evaluates imported functions and their derivatives, such as Ipopt, handles it; a linear
solver such as HiGHS cannot. The example below finds the minimum of $\Gamma(x)$ for $x>0$, a
value with a known answer, $x \approx 1.46163$:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
load amplgsl.dll;
function gsl_sf_gamma;  # Gamma function, with derivatives for the solver

var x >= 1e-5;                          # positive argument
minimize GammaValue: gsl_sf_gamma(x);   # smallest value of Gamma(x)

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.var["x"].set_value(2)
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result, round(ampl.var["x"].value(), 5),
      round(ampl.obj["GammaValue"].value(), 6))

solved 1.46163 0.885603


### Pure AMPL: model.run

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver ipopt;
option ipopt_options "print_level=0";
let x := 2;
solve;
display x, GammaValue;

Ipopt 3.12.13: print_level=0


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
x = 1.46163
GammaValue = 0.885603

